# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdeenMir/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes, backed by two signal checks

**Lane: CTR / Engagement Opportunity Scoring.** Question: which visible pages under-capture
clicks for the position they already hold?

**The rule in plain words:** a page is worth a title/meta review if it sits at a search
position that normally earns a certain click-through rate, but its own CTR falls well short of
that — and it has enough impression volume that the gap is a real signal, not noise from a
handful of searches.

Before coding that, I check the two signals it leans on.

**Signal check A — CTR vs. position (this is the signal behind FlyRank's CTR-fix logic,
i.e. `low_ctr_visible_page` / `needs_ctr_fix`).** Claim: CTR should fall as position gets worse.
Bucket table below, `position_tier`, mean/median CTR, n. `avg_position == 0` rows are dropped
first — the data dictionary flags these as "no position data," and in this slice they were
mis-tagged into the `top_3` tier, which would otherwise fake a spike at the top.

**Signal check B — volume (this is the signal behind FlyRank's quick-win / evidence floor).**
Claim: CTR readings are noisy at low impression volume, so a rule that acts on "CTR looks low"
needs a volume floor before it can be trusted. Bucket table below, impression-volume bucket,
mean/std of CTR, n.

**Reason code (one):** `ctr_below_tier_expected_at_volume` — impressions_90d is at/above the
volume floor, avg_position has real data, and CTR sits below its position tier's own median CTR.

**Action label:** `review_title_meta` when the rule fires above threshold, else `monitor`.

**No future-window or label-derived inputs:** every input here (`avg_position`, `ctr`,
`impressions_90d`, `position_tier`) is a trailing-90-day observed signal, known before any
decision point. `trend_direction` / `trend_pct` (the label source) are never touched.

In [2]:
import pandas as pd
import numpy as np

import os

# Works both in Colab (no local repo checkout -- fetch from GitHub, same pattern as
# w01_research_question.ipynb) and locally after a git clone (relative path, writes
# land inside the real repo tree).
LOCAL_PATH = "../../data/raw/content_refresh_anonymized.csv"
RAW_URL = "https://raw.githubusercontent.com/AdeenMir/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

RUNNING_LOCAL = os.path.exists(LOCAL_PATH)
OUTPUT_DIR = "../outputs" if RUNNING_LOCAL else "work/outputs"  # see cell 2 below for why

df = pd.read_csv(LOCAL_PATH if RUNNING_LOCAL else RAW_URL)
print("rows:", len(df))
print("environment:", "local clone" if RUNNING_LOCAL else "Colab / no local clone")
print("CSV will be written under:", OUTPUT_DIR)

# avg_position == 0 means "no position data" (data dictionary), not rank zero.
# In this slice those rows were mis-tagged into position_tier == "top_3" -- drop before tiering.
has_position = df["avg_position"] > 0
print("rows dropped as no-position-data:", (~has_position).sum())

dpos = df[has_position].copy()

# --- Signal check A: CTR vs. position tier ---
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
signal_a = (
    dpos.groupby("position_tier")["ctr"]
    .agg(mean_ctr="mean", median_ctr="median", n="count")
    .reindex(tier_order)
)
print("\nSignal A -- CTR by position tier (n printed):")
print(signal_a)

means = signal_a["mean_ctr"]
is_monotonic_decreasing = means.is_monotonic_decreasing
print("\nMonotonically decreasing top_3 -> deep:", is_monotonic_decreasing)
verdict_a = "CONFIRMED" if is_monotonic_decreasing else "MIXED"
print(f"Verdict A: {verdict_a}")

# --- Signal check B: CTR reliability by impression volume ---
bins = [0, 100, 500, 3000, 30000, np.inf]
labels = ["<100", "100-500", "500-3000", "3000-30000", "30000+"]
dpos["impr_bucket"] = pd.cut(dpos["impressions_90d"], bins=bins, labels=labels, right=False)

signal_b = (
    dpos.groupby("impr_bucket", observed=True)["ctr"]
    .agg(mean_ctr="mean", std_ctr="std", n="count")
    .reindex(labels)
)
signal_b["cv"] = signal_b["std_ctr"] / signal_b["mean_ctr"]
print("\nSignal B -- CTR mean/std by impression-volume bucket (n printed):")
print(signal_b)

low_bucket_cv = signal_b.loc["<100", "cv"]
mid_bucket_cv = signal_b.loc["500-3000", "cv"]
print(f"\nCoefficient of variation, <100 impressions: {low_bucket_cv:.2f}")
print(f"Coefficient of variation, 500-3000 impressions: {mid_bucket_cv:.2f}")
verdict_b = "CONFIRMED" if low_bucket_cv > mid_bucket_cv * 2 else "MIXED"
print(f"Verdict B: {verdict_b} -- CTR is much noisier under ~500 impressions_90d, "
      f"so the rule's volume floor is set at impressions_90d >= 500.")


rows: 30000
environment: Colab / no local clone
CSV will be written under: work/outputs
rows dropped as no-position-data: 1205

Signal A -- CTR by position tier (n printed):
               mean_ctr  median_ctr      n
position_tier                             
top_3          2.764453        0.00   1116
page_1         0.652467        0.16  11814
striking       0.323239        0.11   7304
page_3_5       0.222484        0.03   7242
deep           0.150212        0.00   1319

Monotonically decreasing top_3 -> deep: True
Verdict A: CONFIRMED

Signal B -- CTR mean/std by impression-volume bucket (n printed):
             mean_ctr   std_ctr     n        cv
impr_bucket                                    
<100         1.370149  6.546444  6789  4.777907
100-500      0.241676  0.596254  5280  2.467160
500-3000     0.216420  0.301543  8443  1.393325
3000-30000   0.308314  0.327929  7205  1.063621
30000+       0.312662  0.307195  1078  0.982513

Coefficient of variation, <100 impressions: 4.78
Coeff

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import os

VOLUME_FLOOR = 500  # justified by Signal check B above

work = df[has_position].copy()

# expected CTR per tier, computed only from rows that clear the volume floor
# (so the "expected" benchmark itself isn't built on noisy low-volume rows)
reliable = work[work["impressions_90d"] >= VOLUME_FLOOR]
expected_ctr_by_tier = reliable.groupby("position_tier")["ctr"].median()
print("Expected CTR by tier (median, volume-floored rows only):")
print(expected_ctr_by_tier)

work["expected_ctr_tier"] = work["position_tier"].map(expected_ctr_by_tier)
work["ctr_gap"] = (work["expected_ctr_tier"] - work["ctr"]).clip(lower=0)
work["meets_volume_floor"] = work["impressions_90d"] >= VOLUME_FLOOR

def normalize(series):
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    lo, hi = values.min(), values.max()
    if hi == lo:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - lo) / (hi - lo)

def percentile_rank(series):
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    return values.rank(method="average", pct=True).fillna(0)

# score = 0 unless the rule's own evidence floor is met
gap_score = normalize(work["ctr_gap"])
volume_weight = percentile_rank(work["impressions_90d"])
work["score"] = np.where(
    work["meets_volume_floor"],
    (gap_score * volume_weight).round(4),
    0.0,
)

work["reason_code"] = "ctr_below_tier_expected_at_volume"

ACTION_THRESHOLD = work.loc[work["meets_volume_floor"], "score"].quantile(0.80)
work["action"] = np.where(work["score"] >= ACTION_THRESHOLD, "review_title_meta", "monitor")
print(f"\nAction threshold (80th pct of eligible scores): {ACTION_THRESHOLD:.4f}")

work["rank"] = work["score"].rank(method="first", ascending=False).astype(int)

out_cols = [
    "content_id", "client_id", "rank", "score", "reason_code", "action",
    "position_tier", "avg_position", "ctr", "expected_ctr_tier", "ctr_gap",
    "impressions_90d", "meets_volume_floor",
]
queue = work[out_cols].sort_values("rank")

os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, "baseline_action_score.csv")
queue.to_csv(out_path, index=False)
print(f"\nWrote {len(queue)} rows to {out_path}")
print(f"Rows flagged review_title_meta: {(queue['action'] == 'review_title_meta').sum()}")

queue.head(10)


Expected CTR by tier (median, volume-floored rows only):
position_tier
deep        0.00
page_1      0.24
page_3_5    0.09
striking    0.17
top_3       0.20
Name: ctr, dtype: float64

Action threshold (80th pct of eligible scores): 0.2967

Wrote 28795 rows to work/outputs/baseline_action_score.csv
Rows flagged review_title_meta: 3346


,content_id,client_id,rank,score,reason_code,action,position_tier,avg_position,ctr,expected_ctr_tier,ctr_gap,impressions_90d,meets_volume_floor
7445,content_c8e9d6ab9013,client_19581e27de,1,0.9990,ctr_below_tier_expected_at_volume,review_title_meta,page_1,9.7,0.00,0.24,0.24,208678,True
27178,content_453722754fea,client_f369cb89fc,2,0.9558,ctr_below_tier_expected_at_volume,review_title_meta,page_1,7.6,0.01,0.24,0.23,140079,True
482,content_39881853ef0c,client_f369cb89fc,3,0.9537,ctr_below_tier_expected_at_volume,review_title_meta,page_1,7.2,0.01,0.24,0.23,112434,True
23220,content_f986bd514b6e,client_7f2253d7e2,4,0.9464,ctr_below_tier_expected_at_volume,review_title_meta,page_1,6.6,0.00,0.24,0.24,22456,True
13631,content_d274ac4158ef,client_4e07408562,5,0.9461,ctr_below_tier_expected_at_volume,review_title_meta,page_1,6.8,0.01,0.24,0.23,65138,True
24866,content_e5f459e737b7,client_f369cb89fc,6,0.9437,ctr_below_tier_expected_at_volume,review_title_meta,page_1,5.9,0.01,0.24,0.23,56363,True
4589,content_339b357d04c7,client_bbb965ab0c,7,0.9395,ctr_below_tier_expected_at_volume,review_title_meta,page_1,3.7,0.01,0.24,0.23,46879,True
3402,content_ca17a024f90c,client_4e07408562,8,0.9338,ctr_below_tier_expected_at_volume,review_title_meta,page_1,9.1,0.01,0.24,0.23,38815,True
25462,content_825a9788af8d,client_4e07408562,9,0.9242,ctr_below_tier_expected_at_volume,review_title_meta,page_1,5.6,0.00,0.24,0.24,16786,True
9443,content_8ba781dafa55,client_8527a891e2,10,0.9211,ctr_below_tier_expected_at_volume,review_title_meta,page_1,9.0,0.00,0.24,0.24,16156,True


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
top10 = queue.head(10).reset_index(drop=True)
pd.set_option("display.max_colwidth", None)
top10_display = top10[[
    "rank", "content_id", "action", "avg_position", "position_tier",
    "ctr", "expected_ctr_tier", "impressions_90d", "score",
]]
print(top10_display.to_string(index=False))

# Plausible reasons a flagged page could still be a bad pick, even though it clears the rule --
# cycled across the top 10 so each row gets a genuine, varied caveat.
wrong_reasons = [
    "the query is navigational (a branded search) -- users find the right page but never click, "
    "since the answer they want is already visible on the SERP.",
    "this is a comparison/listicle page users skim in the snippet and never click through to, "
    "by design of the search intent, not because of a weak title.",
    "avg_position is a 90-day average that could be hiding a page that only recently fell -- "
    "the CTR gap could be new and unrelated to the title at all.",
    "the SERP itself changed for this query (a featured snippet or AI overview absorbing clicks) "
    "-- a title/meta edit would not recover anything.",
    "the query has strong seasonality and this 90-day window just caught a low-demand stretch, "
    "so the gap looks worse than the page's normal performance.",
]

print()
for i, row in top10.iterrows():
    why = wrong_reasons[i % len(wrong_reasons)]
    print(
        f"{int(row['rank'])}. {row['action']} -- {row['position_tier']} tier, "
        f"avg_position {row['avg_position']:.1f}, CTR {row['ctr']:.2f} vs tier-expected "
        f"{row['expected_ctr_tier']:.2f}, on {int(row['impressions_90d']):,} impressions_90d "
        f"(well above the {VOLUME_FLOOR} floor). "
        f"Would be wrong if {why}"
    )


 rank           content_id            action  avg_position position_tier  ctr  expected_ctr_tier  impressions_90d  score
    1 content_c8e9d6ab9013 review_title_meta           9.7        page_1 0.00               0.24           208678 0.9990
    2 content_453722754fea review_title_meta           7.6        page_1 0.01               0.24           140079 0.9558
    3 content_39881853ef0c review_title_meta           7.2        page_1 0.01               0.24           112434 0.9537
    4 content_f986bd514b6e review_title_meta           6.6        page_1 0.00               0.24            22456 0.9464
    5 content_d274ac4158ef review_title_meta           6.8        page_1 0.01               0.24            65138 0.9461
    6 content_e5f459e737b7 review_title_meta           5.9        page_1 0.01               0.24            56363 0.9437
    7 content_339b357d04c7 review_title_meta           3.7        page_1 0.01               0.24            46879 0.9395
    8 content_ca17a024f90c revie

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# Weak picks: same tier, similar gap, but thin evidence relative to the strong picks above
weak_candidates = queue[
    (queue["action"] == "review_title_meta")
].sort_values("score").head(3)
print("Weakest 3 among flagged rows:")
print(weak_candidates[["rank", "content_id", "position_tier", "ctr", "impressions_90d", "score"]]
      .to_string(index=False))
print(
    "\nThese sit right at the volume floor (impressions_90d close to 500) and near the 80th "
    "percentile score cutoff -- a small change in either threshold moves them out of the queue. "
    "Weak, not wrong: they still clear the evidence floor, but with the least margin of any "
    "flagged row."
)

# Leakage check: confirm the label / product-flag columns never entered the score
label_like_cols = {"trend_direction", "trend_pct", "is_declining_label"}
used_cols = {"position_tier", "avg_position", "ctr", "impressions_90d"}
print("\nLabel-like columns used in scoring:", used_cols & label_like_cols)
print("Product-flag columns (health_score, priority_score, action_type, refresh flags) "
      "in this dataset at all:",
      any(c in df.columns for c in
          ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win"]))
print("No future-window columns used (impressions_last_30d / prev_30d untouched):",
      not ({"impressions_last_30d", "impressions_prev_30d"} & used_cols))


Weakest 3 among flagged rows:
 rank           content_id position_tier  ctr  impressions_90d  score
 3346 content_51bb0bff5aed      striking 0.07             2993 0.2967
 3345 content_829eba7c571f      striking 0.00              500 0.2970
 3344 content_cd892ad205d3      striking 0.00              500 0.2970

These sit right at the volume floor (impressions_90d close to 500) and near the 80th percentile score cutoff -- a small change in either threshold moves them out of the queue. Weak, not wrong: they still clear the evidence floor, but with the least margin of any flagged row.

Label-like columns used in scoring: set()
Product-flag columns (health_score, priority_score, action_type, refresh flags) in this dataset at all: False
No future-window columns used (impressions_last_30d / prev_30d untouched): True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.